# Ticketward: training on Kaggle (GPU T4 x2)

A thin wrapper around `python -m tw_ml.train` (spec §9.5). The code and configs come from the
repository at a pinned commit; this notebook holds no training logic. Runbook: `docs/training.md`.

**Before "Save & Run All"**

- Settings: accelerator **GPU T4 x2** and **Internet on** (both need a phone-verified account).
- Add-ons, Secrets: add **`HF_TOKEN`** (write access to your private repos) and attach it to this
  notebook. It goes into the environment only and is never printed.
- Edit the parameters cell: commit SHA, config, private dataset id and revision.

**Steps:** clone at the pinned SHA, platform asserts, pinned installs, the token, train/val data
(sealed splits are never downloaded), the R-21 smoke test (SFT configs), seeds 42 and 1337 in
parallel (one per T4, no DDP), then seed 2026, then push each seed's run files (MLflow store,
epoch adapters, summaries). Checkpoints go to one private Hub repo per seed every 25 steps, and a
rerun resumes from `last-checkpoint/`.

In [ ]:
# Parameters: edit these, then "Save & Run All" (docs/training.md).
REPO_URL = "https://github.com/AkhilMittapalli/Ticketward"
GIT_SHA = "<40-hex commit SHA>"  # the commit whose code and configs this run uses
# CONFIG: sft_qwen35_2b, sft_qwen3_1p7b, sft_qwen3_4b_2507 or encoder_modernbert (ml/configs/)
CONFIG = "ml/configs/sft_qwen35_2b.yaml"
PARALLEL_SEEDS = (42, 1337)  # one per T4 (CUDA_VISIBLE_DEVICES=0 and 1), no DDP
LAST_SEED = 2026
HF_DATASET_ID = "<hf_user>/ticketward-data"  # private dataset; only train/ and val/ are fetched
HF_DATASET_REVISION = "<40-hex dataset commit SHA>"
EXPECTED_TORCH = "2.11"  # platform.torch of the config
GPU_COUNT = 2
RUN_SMOKE_TEST = True  # R-21 smoke test first (SFT configs only)
ALLOW_UNVERIFIED = False  # True only while base revisions are still <verify> (recorded in runs)
RUN_ROOT = "/kaggle/working/tw-runs"  # kept as the notebook's output

In [ ]:
# Clone at the pinned commit and verify it.
import os
import re
import subprocess
import sys

assert re.fullmatch(r"[0-9a-f]{40}", GIT_SHA), "set GIT_SHA to a 40-hex commit SHA"
assert re.fullmatch(r"[0-9a-f]{40}", HF_DATASET_REVISION), "set HF_DATASET_REVISION"
subprocess.run(["git", "clone", "--quiet", REPO_URL, "ticketward"], check=True)
subprocess.run(["git", "-C", "ticketward", "checkout", "--quiet", "--detach", GIT_SHA], check=True)
head = subprocess.run(
    ["git", "-C", "ticketward", "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
assert head == GIT_SHA, f"checked out {head}, expected {GIT_SHA}"
os.chdir("ticketward")
print("repository at", head)

In [ ]:
# The platform: Python 3.12/3.13, the platform torch (never installed by us) and T4 GPUs.
import torch

print(sys.version)
assert sys.version_info[:2] in {(3, 12), (3, 13)}, "tw-ml needs Python 3.12 or 3.13"
torch_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
assert torch_minor == EXPECTED_TORCH, f"torch {torch.__version__}, config expects {EXPECTED_TORCH}"
assert torch.cuda.is_available(), "select a GPU accelerator"
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(torch.__version__, torch.version.cuda, gpus)
assert len(gpus) == GPU_COUNT, f"expected {GPU_COUNT} GPU(s), got {gpus}"
assert all("T4" in name for name in gpus), f"expected T4 GPUs, got {gpus}"

In [ ]:
# Pinned installs: the hashed export of ml/uv.lock (no torch, triton or CUDA wheels), then tw_ml.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "--require-hashes",
        "-r",
        "ml/requirements/train.txt",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "./ml"], check=True
)

In [ ]:
# HF_TOKEN from Kaggle Secrets into the environment only (never printed or logged).
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"], "add the HF_TOKEN secret (Add-ons, Secrets)"

In [ ]:
# Private dataset at the pinned revision: train/ and val/ only.
subprocess.run(
    [
        sys.executable,
        "-m",
        "tw_ml.train.hub",
        "fetch-data",
        "--repo",
        HF_DATASET_ID,
        "--revision",
        HF_DATASET_REVISION,
        "--out",
        "data/generated",
    ],
    check=True,
)

In [ ]:
# R-21 smoke test (SFT configs): fp16, fp32 adapters, loss type, memory, time, resume.
FLAGS = ["--allow-unverified"] if ALLOW_UNVERIFIED else []
os.makedirs(RUN_ROOT, exist_ok=True)
if RUN_SMOKE_TEST and os.path.basename(CONFIG).startswith("sft_"):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "tw_ml.train.smoke",
            "--config",
            CONFIG,
            "--steps",
            "20",
            "--run-root",
            RUN_ROOT,
            *FLAGS,
        ],
        check=True,
        env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
    )

In [ ]:
# Seeds 42 and 1337 in parallel (one per T4, no DDP), then seed 2026.
def train(seed: int, gpu: int) -> subprocess.Popen[bytes]:
    """Start one seed on one GPU; its log goes to RUN_ROOT/train-s<seed>.log."""
    log = open(f"{RUN_ROOT}/train-s{seed}.log", "wb")  # noqa: SIM115 - the process owns it
    return subprocess.Popen(
        [
            sys.executable,
            "-m",
            "tw_ml.train",
            "--config",
            CONFIG,
            "--seed",
            str(seed),
            "--run-root",
            RUN_ROOT,
            *FLAGS,
        ],
        env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
        stdout=log,
        stderr=subprocess.STDOUT,
    )


jobs = {seed: train(seed, gpu) for gpu, seed in enumerate(PARALLEL_SEEDS)}
codes = {seed: job.wait() for seed, job in jobs.items()}
print("exit codes", codes)
assert all(code == 0 for code in codes.values()), f"see {RUN_ROOT}/train-s<seed>.log: {codes}"
assert train(LAST_SEED, 0).wait() == 0, f"see {RUN_ROOT}/train-s{LAST_SEED}.log"

In [ ]:
# Push each seed's run files (MLflow store, epoch adapters, choice, summaries) again.
for seed in (*PARALLEL_SEEDS, LAST_SEED):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "tw_ml.train.hub",
            "push-run",
            "--config",
            CONFIG,
            "--seed",
            str(seed),
            "--run-root",
            RUN_ROOT,
        ],
        check=True,
    )